# Demo: ComBat local priors (directional-bias weighting)

This notebook demonstrates `combat_modular(..., prior_mode="local")`
using synthetic data where batch effects vary smoothly across features.
We compare `prior_mode='global'` vs `prior_mode='local'` and show priors/weights.

In [1]:
import numpy as np
import pandas as pd
from DiagnoseHarmonisation.HarmonisationFunctions import combat_modular
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'DiagnoseHarmonisation'

In [ ]:
# Synthetic data setup
np.random.seed(0)
n_features = 50
n_samples = 120
# features arranged on a 1D spatial axis
coords = np.linspace(0, 1, n_features)[:, None]
# ground truth signal: smooth feature-wise amplitude varying across features
true_feature_amp = np.sin(2 * np.pi * coords[:, 0]) * 0.5 + 1.0
# sample covariate (age-like) with nonlinear effect
age = np.linspace(0, 2 * np.pi, n_samples)
signal = np.outer(true_feature_amp, np.sin(age))
# batch assignment and batch effect varying by feature (spatially smooth)
batch = np.random.choice(["A", "B"], size=n_samples)
batch_effect_profile = np.cos(4 * np.pi * coords[:, 0]) * 0.6
batch_effect = np.where(batch == 'A', batch_effect_profile[:, None], -batch_effect_profile[:, None])
# data = signal + batch effect + noise
data = signal + batch_effect + 0.2 * np.random.randn(n_features, n_samples)
data_df = pd.DataFrame(data)
mod = pd.DataFrame({"age": age})

In [ ]:
# Run global priors (baseline)
out_global = combat_modular(data=data_df, batch=pd.Series(batch), mod=mod, mean_model='gam', prior_mode='global', return_priors=True)
# Run local priors with directional bias and spatial coord weighting
out_local = combat_modular(
    data=data_df,
    batch=pd.Series(batch),
    mod=mod,
    mean_model='gam',
    prior_mode='local',
    prior_weight_methods=['spatial_proximity','directional_bias'],
    prior_weight_opts={'feature_coords': coords, 'method_weights': {'spatial_proximity':0.4, 'directional_bias':0.6}, 'directional':{'dir_sigma':0.1,'dir_power':1.2}},
    return_priors=True,
)

In [ ]:
# Visualize a sample of the local weight matrix for batch 0
local = out_local['priors']['local_priors']
W0 = local['weights'][0]
plt.figure(figsize=(4,4))
plt.imshow(W0, cmap='viridis', aspect='auto')
plt.title('Local weight matrix (batch 0)')
plt.xlabel('feature j')
plt.ylabel('feature i')
plt.colorbar()
plt.show()

In [ ]:
# Compare MSE to ground truth signal for baseline vs local-prior harmonisation
bayes_global = out_global['bayesdata']
bayes_local = out_local['bayesdata']
# Convert to arrays if returned as dict or df
if isinstance(bayes_global, dict):
    bayes_global = bayes_global['bayesdata']
if isinstance(bayes_local, dict):
    bayes_local = bayes_local['bayesdata']
arr_g = np.asarray(bayes_global)
arr_l = np.asarray(bayes_local)
mse_global = np.mean((arr_g - signal) ** 2)
mse_local = np.mean((arr_l - signal) ** 2)
print(f"MSE global: {mse_global:.4f}, MSE local: {mse_local:.4f}")

If `MSE local < MSE global` then local priors improved recovery of the true signal in this toy example.
Adjust `prior_weight_methods` and options to tune behaviour; for large feature sets consider using a sparse kNN-based weight matrix to reduce cost.